In [ ]:
import numpy as np
import pandas as pd

def _encode_municipality(df, le_municipality):
    df['municipality_encoded'] = le_municipality.fit_transform(df['municipality'])

    df = df.drop(columns='municipality')

    return df

def _get_season(month):
    if month in [12, 1, 2]:
        return 0
    if month in [3, 4, 5]:
        return 1
    if month in [6, 7, 8]:
        return 2
    else:
        return 3

def _extract_datetime_features(df):
    df['year'] = df['date_time'].dt.year
    df['month'] = df['date_time'].dt.month
    df['day_of_week'] = df['date_time'].dt.dayofweek
    df['hour'] = df['date_time'].dt.hour
    df['day_type'] = np.where((df['day_of_week'] == 5) | (df['day_of_week'] == 6), 1, 0)
    df['season'] = df['month'].apply(_get_season)
    df['is_rush'] = df['hour'].isin([7, 8, 16, 17, 18]).astype(int)
    df['is_night'] = (df['hour'].between(22, 23) | df['hour'].between(0, 6)).astype(int)

    df = df.drop(columns='date_time')

    return df

def _encode_involved_vehicles(df):
    mapping = {
        r'SN SA JEDNIM VOZILOM': 'single_vehicle',
        r'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA': 'two_vehicles_no_turn',
        r'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK': 'two_vehicles_turn_or_cross',
        r'SN SA PARKIRANIM VOZILIMA': 'parked_vehicles',
        r'SN SA PEŠACIMA': 'pedestrians' 
    }

    df['involved_vehicles_num'] = df['involved_vehicles_num'].astype(str).str.strip().replace(mapping, regex=True)

    dummies = pd.get_dummies(df['involved_vehicles_num'], prefix='acc', dtype=int)

    df = pd.concat([df, dummies], axis=1)

    df = df.drop(columns='involved_vehicles_num')

    return df

def _encode_description(df, le_description):
    df['description_encoded'] = le_description.fit_transform(df['description'])

    df = df.drop(columns='description')

    return df

def _encode_target_accident_type(df):
    mapping = {
        'Sa mat.stetom': 'material',
        'Sa povredjenim': 'injured',
        'Sa poginulim': 'dead',
    }

    df['accident_type'] = df['accident_type'].astype(str).str.strip()
    df['accident_type'] = df['accident_type'].replace(mapping)
    
    df['accident_type'] = (df['accident_type'] != 'material').astype(int)

    return df

def _drop_unused_columns(df):
    columns = ['accident_id', 'department', 'longitude', 'latitude', 'year']
    
    df = df.drop(columns=columns)

    return df

def preprocess(df: pd.DataFrame, le_municipality, le_description) -> pd.DataFrame:
    df = df.copy()

    df = _encode_municipality(df, le_municipality)
    df = _extract_datetime_features(df)
    df = _encode_involved_vehicles(df)
    df = _encode_description(df, le_description)
    df = _encode_target_accident_type(df)
    df = _drop_unused_columns(df)

    return df

In [22]:
data = pd.read_excel('../data/raw/nez-opendata-all.xlsx')

KeyboardInterrupt: 

In [16]:
import joblib

In [17]:
le_municipality = joblib.load('../models/encoders/le_municipality.pkl')
le_description = joblib.load('../models/encoders/le_municipality.pkl')

c:\Users\iam0v\Desktop\PROJECTS\road-pulse\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [21]:
data = preprocess(data, le_municipality, le_description)

KeyError: 'municipality'

In [ ]:
data.head()

,accident_type,municipality_encoded,year,month,day_of_week,hour,day_type,season,is_rush,is_night,acc_SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,acc_SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,description_encoded
0,0,9,2020,1,1,10,0,0,0,0,0,0,0,0,1,24
1,0,9,2020,1,3,12,0,0,0,0,1,0,0,0,0,17
2,0,9,2020,1,4,10,0,0,0,0,0,1,0,0,0,7
3,0,9,2020,1,4,15,0,0,0,0,0,1,0,0,0,10
4,1,9,2020,1,3,16,0,0,1,0,1,0,0,0,0,16
